# Time Series AIME: APR experiments v9.3

v9.2 is retained unchanged. This notebook contains all APR-specific code.
Only generic inverse operators, S-Map, and calendar statistics come from `tsaime`.

Run every cell in order in a fresh local Jupyter kernel. No GitHub installation is required when the `codev9` folder is present.
Default mode is `publication` (12 Beijing sites plus Tsukuba). Choose `smoke` only to check execution.
Figures (PDF/PNG), full CSVs, explicit TeX summaries, logs, and a ZIP are generated in a NEW timestamped `results/v9_3_apr/` directory.
Existing downloaded source data are reused from `results/v9_apr/data/`.
Private per-origin reproduction traces are saved outside the ZIP because they contain source observations.

This protocol was fixed after inspecting v9.2. It is not preregistered unseen confirmation.
PASS is a computational check, not a favorable scientific result or a publication guarantee.
The NIES license-notice discrepancy remains a separate manual release check.


## Mathematical and interpretation contract

With fit-only centered/standardized inputs $X$ and forecast outputs $Y$,
$$\widehat A_\lambda=\arg\min_A\{n^{-1}\|X-YA^\top\|_F^2+\lambda\|A\|_F^2\}
=S_{XY}(S_{YY}+\lambda I)^\dagger.$$
The matrix is a regularized linear inverse reconstruction, not an exact nonlinear inverse or a causal attribution.
For one standardized output and zero ridge it equals Pearson correlation.
All inverse tuning uses standardized input error; quadratic Ridge uses alpha = n * lambda.
The single horizon is selected using validation data, never the test minimum.

S-Map fits the local forward direction; ts-AIME reconstructs input information from the joint forecast vector.
The forward library remains before test, while the rolling decoder adapts using past test-period forecast/input pairs.
Three forecast coordinates cannot uniquely identify an arbitrary ten-dimensional atmospheric state.
Current-PM2.5 reconstruction is separated from meteorological reconstruction in the figures.

For $Y=FX+r$, $C=\mathrm{Cov}(X,r)$,
$$S_{XY}=\Sigma_XF^\top+C,\quad S_{YY}=F\Sigma_XF^\top+FC+C^\top F^\top+\Sigma_r.$$
The uncorrelated-residual approximation requires $C=0$; restoring C is an algebraic identity, not independent validation.
Intervals preserve missing calendar positions. The primary 28-day block and 7/14-day sensitivity intervals are pointwise and conditional on fitted predictions.
Complete-week circular-shift tail ranks do not remove seasonal confounding or guarantee exact p-values.


## 1. Local installation and paths

In [ ]:
from __future__ import annotations

import importlib
import importlib.metadata
import os
os.environ.setdefault("MPLBACKEND", "Agg")
import subprocess
import sys
from pathlib import Path

import pandas as pd


def locate_tsaime_source() -> Path | None:
    explicit = os.environ.get("TSAIME_V3_ROOT") or os.environ.get("TSAIME_CODEV9_ROOT")
    candidates = []
    if explicit:
        candidates.append(Path(explicit).expanduser())
    current = Path.cwd().resolve()
    candidates.extend([current, *current.parents, current / "codev9", current.parent / "codev9"])
    for candidate in candidates:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "tsaime").is_dir():
            return candidate.resolve()
    return None


PACKAGE_ROOT = locate_tsaime_source()
try:
    installed_version = importlib.metadata.version("tsaime")
except importlib.metadata.PackageNotFoundError:
    installed_version = None

if PACKAGE_ROOT is not None:
    local_source = str(PACKAGE_ROOT / "src")
    if local_source not in sys.path:
        sys.path.insert(0, local_source)
elif installed_version != "0.3.2":
    raise RuntimeError(
        "tsaime 0.3.2 is required. Install the released wheel or GitHub repository, "
        "then restart the kernel."
    )

APR_REQUIREMENTS = {
    "matplotlib": "matplotlib>=3.7",
    "pyEDM": "pyEDM==2.5.7",
    "sklearn": "scikit-learn>=1.2",
}
missing_requirements = [
    requirement
    for module, requirement in APR_REQUIREMENTS.items()
    if importlib.util.find_spec(module) is None
]
if missing_requirements:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *missing_requirements]
    )
    importlib.invalidate_caches()

from tsaime import __version__

if __version__ != "0.3.2":
    raise RuntimeError(f"Expected tsaime 0.3.2, found {__version__}. Restart the kernel after installation.")

RESULT_BASE_DIR = Path(
    os.environ.get("TSAIME_V9_BASE_DIR", PACKAGE_ROOT or Path.cwd())
).expanduser().resolve()

print("tsaime:", __version__)
print("tsaime source:", PACKAGE_ROOT if PACKAGE_ROOT is not None else "installed package")
print("result base:", RESULT_BASE_DIR)


## 2. Atmospheric preprocessing

In [ ]:
"""Atmospheric preprocessing fixed for the APR PM2.5 v9 experiment."""

from __future__ import annotations

from typing import Sequence

import numpy as np
import pandas as pd

from tsaime.temporal import (
    ChronologicalSplit,
    complete_hourly,
    prepare_chronological_pairs,
)


DEFAULT_STATE_COLUMNS = (
    "PM25_t",
    "PM25_lag1",
    "PM25_lag24",
    "TEMP",
    "RH",
    "PRESS",
    "RAIN_EVENT",
    "LOG_RAIN",
    "WIND_U",
    "WIND_V",
)


PM25_POLICIES = ("as_reported", "negative_missing", "negative_zero")


def apply_pm25_policy(
    frame: pd.DataFrame, policy: str, *, pm25_column: str = "PM25"
) -> pd.DataFrame:
    """Apply an explicit PM2.5 negative-value policy without interpolation."""

    if policy not in PM25_POLICIES:
        raise ValueError(f"unknown PM2.5 policy: {policy}")
    data = frame.copy()
    if pm25_column not in data:
        raise KeyError(f"missing PM2.5 column: {pm25_column}")
    if policy == "negative_missing":
        data.loc[data[pm25_column] < 0, pm25_column] = np.nan
    elif policy == "negative_zero":
        data.loc[data[pm25_column] < 0, pm25_column] = 0.0
    return data


def pm25_plausibility_audit(
    frame: pd.DataFrame, *, dataset: str, site: str
) -> dict[str, object]:
    """Record PM2.5 range and negative observations before any policy is applied."""

    values = pd.to_numeric(frame["PM25"], errors="coerce")
    finite = values[np.isfinite(values)]
    temperature = pd.to_numeric(frame["TEMP"], errors="coerce")
    humidity = pd.to_numeric(frame["RH"], errors="coerce")
    pressure = pd.to_numeric(frame["PRESS"], errors="coerce")
    rain = pd.to_numeric(frame["RAIN"], errors="coerce")
    wind_speed = np.sqrt(pd.to_numeric(frame["WIND_U"], errors="coerce") ** 2 + pd.to_numeric(frame["WIND_V"], errors="coerce") ** 2)
    return {
        "dataset": dataset,
        "site": site,
        "pm25_nonmissing_n": int(finite.size),
        "pm25_negative_n": int((finite < 0).sum()),
        "pm25_zero_n": int((finite == 0).sum()),
        "pm25_min": float(finite.min()) if finite.size else np.nan,
        "pm25_max": float(finite.max()) if finite.size else np.nan,
        "temperature_min_c": float(temperature.min()),
        "temperature_max_c": float(temperature.max()),
        "rh_outside_0_100_n": int(((humidity < 0) | (humidity > 100)).sum()),
        "pressure_outside_800_1100_n": int(((pressure < 800) | (pressure > 1100)).sum()),
        "negative_rain_n": int((rain < 0).sum()),
        "wind_speed_max_m_s": float(wind_speed.max()),
        "main_policy": "as_reported",
        "sensitivity_policies": "negative_missing; negative_zero",
    }


def relative_humidity_from_dewpoint(
    temperature_c: Sequence[float] | pd.Series,
    dewpoint_c: Sequence[float] | pd.Series,
) -> np.ndarray:
    """Approximate relative humidity using the Magnus relation."""

    temperature = np.asarray(temperature_c, dtype=float)
    dewpoint = np.asarray(dewpoint_c, dtype=float)
    if temperature.shape != dewpoint.shape:
        raise ValueError("temperature and dew point must have the same shape")
    numerator = np.exp((17.625 * dewpoint) / (243.04 + dewpoint))
    denominator = np.exp((17.625 * temperature) / (243.04 + temperature))
    return np.clip(100.0 * numerator / denominator, 0.0, 100.0)


def wind_components(
    speed: Sequence[float] | pd.Series,
    direction_degrees: Sequence[float] | pd.Series,
) -> tuple[np.ndarray, np.ndarray]:
    """Convert meteorological FROM-direction and speed to east/north components."""

    speed_array = np.asarray(speed, dtype=float)
    direction = np.asarray(direction_degrees, dtype=float)
    if speed_array.shape != direction.shape:
        raise ValueError("speed and direction must have the same shape")
    theta = np.deg2rad(direction)
    return -speed_array * np.sin(theta), -speed_array * np.cos(theta)


def engineer_pm25_multihorizon_frame(
    frame: pd.DataFrame,
    horizons: Sequence[int],
    *,
    pm25_column: str = "PM25",
    date_column: str = "Date",
) -> tuple[pd.DataFrame, list[str], list[str]]:
    """Create the APR-specific PM2.5 state and target columns on an hourly grid."""

    horizon_values = tuple(int(value) for value in horizons)
    if not horizon_values or any(value < 1 for value in horizon_values):
        raise ValueError("horizons must contain positive integers")
    if len(set(horizon_values)) != len(horizon_values):
        raise ValueError("horizons must not contain duplicates")
    data = complete_hourly(frame, date_column=date_column).set_index(date_column).sort_index()
    if pm25_column not in data:
        raise KeyError(f"missing PM2.5 column: {pm25_column}")
    data["PM25_t"] = data[pm25_column]
    data["PM25_lag1"] = data[pm25_column].shift(1)
    data["PM25_lag24"] = data[pm25_column].shift(24)
    if "RAIN" in data:
        data["RAIN_EVENT"] = np.where(
            data["RAIN"].notna(), (data["RAIN"] > 0).astype(float), np.nan
        )
        data["LOG_RAIN"] = np.log1p(data["RAIN"].clip(lower=0))
    targets = []
    for horizon in horizon_values:
        target = f"PM25_future_{horizon}h"
        data[target] = data[pm25_column].shift(-horizon)
        targets.append(target)
    states = [column for column in DEFAULT_STATE_COLUMNS if column in data]
    return data.reset_index(), states, targets


def prepare_pm25_multihorizon_pairs(
    frame: pd.DataFrame,
    horizons: Sequence[int],
    split: ChronologicalSplit,
    *,
    origin_step_hours: int = 1,
    required_states: Sequence[str] | None = None,
    date_column: str = "Date",
) -> tuple[pd.DataFrame, pd.DataFrame, list[str], list[str]]:
    """Build exact-time PM2.5 pairs and apply the generic chronological split."""

    horizon_values = tuple(int(value) for value in horizons)
    engineered, states, targets = engineer_pm25_multihorizon_frame(
        frame,
        horizon_values,
        date_column=date_column,
    )
    if required_states is not None:
        states = [str(value) for value in required_states]
        missing = [column for column in states if column not in engineered]
        if missing:
            raise KeyError(f"missing required state columns: {missing}")
    candidates, complete = prepare_chronological_pairs(
        engineered,
        states,
        targets,
        split,
        max_target_horizon_hours=max(horizon_values),
        origin_step_hours=origin_step_hours,
        date_column=date_column,
    )
    return candidates, complete, states, targets


## 3. Source acquisition, DOI and licenses

In [ ]:
"""APR-specific public data loaders; this module is not part of ``tsaime``."""

from __future__ import annotations

import gzip
import hashlib
import io
import shutil
import urllib.parse
import urllib.request
import zipfile
import xml.etree.ElementTree as ET
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, field
from pathlib import Path
from typing import Mapping, Sequence

import numpy as np
import pandas as pd

from tsaime.temporal import complete_hourly



UCI_BEIJING_URL = "https://archive.ics.uci.edu/static/public/501/beijing%2Bmulti%2Bsite%2Bair%2Bquality%2Bdata.zip"
UCI_BEIJING_LANDING_PAGE = "https://archive.ics.uci.edu/dataset/501/beijingmultisiteairqualitydata"
UCI_BEIJING_DOI = "10.24432/C5RK5G"
NIES_LANDING_PAGE = "https://db.cger.nies.go.jp/MD/10.17595/20250418.001.html.en"
NIES_DOI = "10.17595/20250418.001"
NIES_VERSION = "1.0"
NIES_TEMPLATE = "https://db.cger.nies.go.jp/nies_data/10.17595/20250418.001/AMEL.hourly.{year}.Ver1.0.txt"
SOURCE_ACCESSED_ON = "2026-09-04"
OPENAQ_BUCKET = "https://openaq-data-archive.s3.amazonaws.com/"
NASA_POWER_ENDPOINT = "https://power.larc.nasa.gov/api/temporal/hourly/point"

THAI_SITES: dict[str, dict[str, float | int]] = {
    "Chonburi": {"location_id": 225654, "latitude": 13.35461667, "longitude": 100.9792167},
    "Chiang_Mai": {"location_id": 225669, "latitude": 18.840732, "longitude": 98.96978},
}

COMPASS_DEGREES = {
    "N": 0.0, "NNE": 22.5, "NE": 45.0, "ENE": 67.5,
    "E": 90.0, "ESE": 112.5, "SE": 135.0, "SSE": 157.5,
    "S": 180.0, "SSW": 202.5, "SW": 225.0, "WSW": 247.5,
    "W": 270.0, "WNW": 292.5, "NW": 315.0, "NNW": 337.5,
}


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


@dataclass
class PublicDataRepository:
    """Download and cache the public datasets used by the reference workflow."""

    data_dir: Path | str
    user_agent: str = "tsAIME-v9-research/1.0"
    records: list[dict[str, object]] = field(default_factory=list)
    license_records: list[dict[str, object]] = field(default_factory=list)

    def __post_init__(self) -> None:
        self.data_dir = Path(self.data_dir).expanduser().resolve()
        self.data_dir.mkdir(parents=True, exist_ok=True)

    def download(self, url: str, relative_path: str | Path, timeout: int = 180) -> Path:
        destination = self.data_dir / relative_path
        destination.parent.mkdir(parents=True, exist_ok=True)
        cached = destination.exists() and destination.stat().st_size > 0
        if not cached:
            request = urllib.request.Request(url, headers={"User-Agent": self.user_agent})
            temporary = destination.with_suffix(destination.suffix + ".part")
            with urllib.request.urlopen(request, timeout=timeout) as response, temporary.open("wb") as handle:
                shutil.copyfileobj(response, handle)
            temporary.replace(destination)
        self.records.append(
            {
                "url": url,
                "path": str(destination.relative_to(self.data_dir)),
                "bytes": destination.stat().st_size,
                "sha256": _sha256(destination),
                "cached": cached,
                "file_mtime_utc": datetime.fromtimestamp(
                    destination.stat().st_mtime, tz=timezone.utc
                ).isoformat(),
                "verified_in_run_utc": datetime.now(timezone.utc).isoformat(),
            }
        )
        return destination

    def load_beijing_sites(self, limit: int = 12) -> dict[str, pd.DataFrame]:
        """Load UCI Beijing Multi-Site data, including the current nested ZIP."""

        archive = self.download(UCI_BEIJING_URL, "uci_beijing_multisite.zip")
        payloads: list[tuple[str, bytes]] = []
        with zipfile.ZipFile(archive) as outer:
            nested = sorted(name for name in outer.namelist() if name.lower().endswith(".zip"))
            if nested:
                with zipfile.ZipFile(io.BytesIO(outer.read(nested[0]))) as source:
                    members = sorted(
                        name for name in source.namelist()
                        if name.lower().endswith(".csv") and not Path(name).name.startswith("._")
                    )
                    payloads = [(name, source.read(name)) for name in members[:limit]]
            else:
                members = sorted(
                    name for name in outer.namelist()
                    if name.lower().endswith(".csv") and not Path(name).name.startswith("._")
                )
                payloads = [(name, outer.read(name)) for name in members]
        required = {"year", "month", "day", "hour", "PM2.5", "TEMP", "DEWP", "PRES", "RAIN", "wd", "WSPM"}
        sites: dict[str, pd.DataFrame] = {}
        for member, payload in payloads:
            raw = pd.read_csv(io.BytesIO(payload))
            if not required.issubset(raw.columns):
                continue
            raw["Date"] = pd.to_datetime(
                dict(year=raw["year"], month=raw["month"], day=raw["day"], hour=raw["hour"])
            )
            site = (
                str(raw["station"].dropna().iloc[0])
                if "station" in raw and raw["station"].notna().any()
                else Path(member).stem
            )
            degrees = raw["wd"].astype(str).str.upper().map(COMPASS_DEGREES)
            u, v = wind_components(raw["WSPM"], degrees)
            frame = pd.DataFrame(
                {
                    "Date": raw["Date"],
                    "PM25": pd.to_numeric(raw["PM2.5"], errors="coerce"),
                    "TEMP": pd.to_numeric(raw["TEMP"], errors="coerce"),
                    "RH": relative_humidity_from_dewpoint(raw["TEMP"], raw["DEWP"]),
                    "PRESS": pd.to_numeric(raw["PRES"], errors="coerce"),
                    "RAIN": pd.to_numeric(raw["RAIN"], errors="coerce"),
                    "WIND_U": u,
                    "WIND_V": v,
                }
            )
            sites[site] = complete_hourly(frame)
        if not sites:
            raise RuntimeError("no valid Beijing station CSV was found")
        return sites

    def load_tsukuba(self, years: Sequence[int] = tuple(range(2017, 2023))) -> pd.DataFrame:
        frames = []
        for year in years:
            path = self.download(
                NIES_TEMPLATE.format(year=int(year)),
                f"AMEL.hourly.{int(year)}.Ver1.0.txt",
            )
            header = path.read_text(encoding="utf-8", errors="replace").splitlines()[:100]
            embedded = next((line.strip(" -") for line in header if "CC BY" in line), "not found")
            self.license_records.append(
                {
                    "dataset": "Tsukuba", "file": path.name,
                    "landing_page_license": "CC BY-NC-ND 4.0",
                    "embedded_file_notice": embedded,
                    "accessed_on": SOURCE_ACCESSED_ON,
                    "status": "conflict: verify controlling publisher terms before release",
                }
            )
            frames.append(_parse_nies_year(path))
        return complete_hourly(pd.concat(frames, ignore_index=True))

    def load_thai_site(
        self,
        name: str,
        metadata: Mapping[str, float | int],
        years: Sequence[int],
        *,
        workers: int = 4,
    ) -> pd.DataFrame:
        pm25 = self._load_openaq_pm25(int(metadata["location_id"]), years, workers=workers)
        meteorology = self._load_nasa_power(
            float(metadata["latitude"]),
            float(metadata["longitude"]),
            f"{min(years)}0101",
            f"{max(years)}1231",
            name,
        )
        merged = pm25.merge(meteorology, on="DateUTC", how="outer").sort_values("DateUTC")
        merged["Date"] = merged["DateUTC"].dt.tz_convert("Asia/Bangkok").dt.tz_localize(None)
        return complete_hourly(merged.drop(columns="DateUTC"))

    def _openaq_keys(self, location_id: int, years: Sequence[int]) -> list[str]:
        prefix = f"records/csv.gz/locationid={location_id}/"
        token: str | None = None
        keys: list[str] = []
        while True:
            params = {"list-type": "2", "prefix": prefix}
            if token:
                params["continuation-token"] = token
            request = urllib.request.Request(
                OPENAQ_BUCKET + "?" + urllib.parse.urlencode(params),
                headers={"User-Agent": self.user_agent},
            )
            with urllib.request.urlopen(request, timeout=180) as response:
                root = ET.fromstring(response.read())
            namespace = {"s3": "http://s3.amazonaws.com/doc/2006-03-01/"}
            for node in root.findall("s3:Contents/s3:Key", namespace):
                key = node.text or ""
                if any(f"/year={int(year)}/" in key for year in years):
                    keys.append(key)
            truncated = root.findtext(
                "s3:IsTruncated", default="false", namespaces=namespace
            ).lower() == "true"
            if not truncated:
                break
            token = root.findtext(
                "s3:NextContinuationToken", default="", namespaces=namespace
            )
            if not token:
                break
        return sorted(keys)

    def _load_openaq_pm25(
        self, location_id: int, years: Sequence[int], *, workers: int
    ) -> pd.DataFrame:
        keys = self._openaq_keys(location_id, years)
        if not keys:
            raise RuntimeError(f"no OpenAQ objects found for location {location_id}")

        def fetch(key: str) -> Path:
            relative = Path(f"openaq_{location_id}") / key.replace("/", "__")
            return self.download(
                OPENAQ_BUCKET + urllib.parse.quote(key, safe="/="), relative, timeout=120
            )

        paths: list[Path] = []
        with ThreadPoolExecutor(max_workers=max(1, workers)) as executor:
            futures = [executor.submit(fetch, key) for key in keys]
            for future in as_completed(futures):
                try:
                    paths.append(future.result())
                except Exception:
                    continue
        rows = []
        for path in sorted(paths):
            try:
                daily = pd.read_csv(path, compression="gzip")
                parameter = "parameter" if "parameter" in daily else next(
                    column for column in daily if "parameter" in column.lower()
                )
                date = "datetime" if "datetime" in daily else next(
                    column for column in daily if "date" in column.lower()
                )
                value = "value" if "value" in daily else next(
                    column for column in daily if column.lower() == "value"
                )
                normalized = daily[parameter].astype(str).str.lower().str.replace(".", "", regex=False)
                selected = daily.loc[normalized.isin({"pm25", "pm₂5"}), [date, value]].copy()
                selected.columns = ["DateUTC", "PM25"]
                rows.append(selected)
            except Exception:
                continue
        if not rows:
            raise RuntimeError(f"OpenAQ objects for {location_id} contained no PM2.5 rows")
        data = pd.concat(rows, ignore_index=True)
        data["DateUTC"] = pd.to_datetime(data["DateUTC"], utc=True, errors="coerce").dt.floor("h")
        data["PM25"] = pd.to_numeric(data["PM25"], errors="coerce")
        data.loc[data["PM25"] < 0, "PM25"] = np.nan
        return data.groupby("DateUTC", as_index=False)["PM25"].mean()

    def _load_nasa_power(
        self, latitude: float, longitude: float, start: str, end: str, name: str
    ) -> pd.DataFrame:
        query = urllib.parse.urlencode(
            {
                "parameters": "T2M,RH2M,PS,PRECTOT,U10M,V10M",
                "community": "SB",
                "longitude": longitude,
                "latitude": latitude,
                "start": start,
                "end": end,
                "format": "CSV",
                "time-standard": "UTC",
            }
        )
        safe_name = "".join(character if character.isalnum() else "_" for character in name)
        path = self.download(
            f"{NASA_POWER_ENDPOINT}?{query}",
            f"nasa_power_{safe_name}_{start}_{end}.csv",
        )
        lines = path.read_text(encoding="utf-8", errors="replace").splitlines()
        header_end = next(index for index, line in enumerate(lines) if "-END HEADER-" in line)
        data = pd.read_csv(io.StringIO("\n".join(lines[header_end + 1 :])))
        data["DateUTC"] = pd.to_datetime(
            dict(year=data["YEAR"], month=data["MO"], day=data["DY"], hour=data["HR"]),
            utc=True,
        )
        for column in ["T2M", "RH2M", "PS", "PRECTOT", "U10M", "V10M"]:
            data[column] = pd.to_numeric(data[column], errors="coerce").mask(
                lambda values: values <= -900
            )
        data["PS"] = 10.0 * data["PS"]  # NASA POWER kPa to common hPa
        return data.rename(
            columns={
                "T2M": "TEMP",
                "RH2M": "RH",
                "PS": "PRESS",
                "PRECTOT": "RAIN",
                "U10M": "WIND_U",
                "V10M": "WIND_V",
            }
        )[["DateUTC", "TEMP", "RH", "PRESS", "RAIN", "WIND_U", "WIND_V"]]


NIES_COLUMNS = [
    "TIME", "YEAR", "MONTH", "DAY", "HOUR", "CH4", "NMHC", "O3", "SO2", "NO",
    "NO2", "SPM", "PM25", "TEMP", "RH", "PRESS", "SR", "UVA", "RAIN", "WD", "WS",
]
NIES_SENTINELS = {
    "CH4": 9.99, "NMHC": 9.99, "O3": 999, "SO2": 999, "NO": 999, "NO2": 999,
    "SPM": 999, "PM25": 9999, "TEMP": 99.9, "RH": 999, "PRESS": 9999,
    "SR": 9.99, "UVA": 999, "RAIN": 999.9, "WD": 99, "WS": 99.9,
}


def _parse_nies_year(path: Path) -> pd.DataFrame:
    with path.open("r", encoding="utf-8", errors="replace") as handle:
        header_lines = int(handle.readline().split()[0])
    raw = pd.read_csv(
        path,
        sep=r"\s+",
        skiprows=header_lines,
        names=NIES_COLUMNS,
        engine="python",
        on_bad_lines="skip",
    )
    for column in NIES_COLUMNS:
        raw[column] = pd.to_numeric(raw[column], errors="coerce")
    for column, sentinel in NIES_SENTINELS.items():
        raw.loc[np.isclose(raw[column], sentinel, equal_nan=False), column] = np.nan
    base = pd.to_datetime(
        dict(year=raw["YEAR"], month=raw["MONTH"], day=raw["DAY"]), errors="coerce"
    )
    raw["Date"] = base + pd.to_timedelta(raw["HOUR"], unit="h")
    code = raw["WD"]
    degrees = ((code % 16.0) * 22.5).where(code.between(1, 16))
    u, v = wind_components(raw["WS"], degrees)
    calm = (code == 0) | (raw["WS"] < 0.4)
    u = pd.Series(u).mask(calm, 0.0).to_numpy()
    v = pd.Series(v).mask(calm, 0.0).to_numpy()
    return pd.DataFrame(
        {
            "Date": raw["Date"],
            "PM25": raw["PM25"],
            "TEMP": raw["TEMP"],
            "RH": raw["RH"],
            "PRESS": raw["PRESS"],
            "RAIN": raw["RAIN"],
            "WIND_U": u,
            "WIND_V": v,
        }
    ).dropna(subset=["Date"])


def source_table() -> pd.DataFrame:
    """Return the prespecified data-source roles for reporting."""

    return pd.DataFrame(
        [
            {
                "dataset": "Beijing",
                "provider": "UCI / Beijing Municipal Environmental Monitoring Center",
                "period_used": "2013-03-01 to 2017-02-28",
                "resolution": "hourly",
                "sites_planned": 12,
                "doi": UCI_BEIJING_DOI,
                "version": "UCI dataset 501 (download snapshot)",
                "accessed_on": SOURCE_ACCESSED_ON,
                "landing_page": UCI_BEIJING_LANDING_PAGE,
                "license_or_terms": "CC BY 4.0",
                "redistribution": "raw data excluded from result ZIP",
                "role": "required multi-site analysis",
            },
            {
                "dataset": "Tsukuba",
                "provider": "National Institute for Environmental Studies (NIES)",
                "period_used": "2017-01-01 to 2022-12-31",
                "resolution": "hourly",
                "sites_planned": 1,
                "doi": NIES_DOI,
                "version": NIES_VERSION,
                "accessed_on": SOURCE_ACCESSED_ON,
                "landing_page": NIES_LANDING_PAGE,
                "license_or_terms": "landing page: CC BY-NC-ND 4.0; downloaded text header observed as CC BY 4.0; recheck current publisher terms before release",
                "redistribution": "no raw redistribution",
                "role": "required external-region validation",
            },
            {
                "dataset": "Thailand",
                "provider": "Air4Thai via OpenAQ + NASA POWER",
                "period_used": "2021-2025 when available",
                "resolution": "hourly",
                "sites_planned": 2,
                "doi": "not assigned for combined optional input",
                "version": "source snapshot at run time",
                "accessed_on": "record when extension is enabled",
                "landing_page": "https://openaq.org/; https://power.larc.nasa.gov/",
                "license_or_terms": "source terms must be verified when enabled",
                "redistribution": "no raw redistribution",
                "role": "optional quality-gated extension",
            },
        ]
    )


## 4. Fixed periods and configuration

In [ ]:
"""APR-specific PM2.5 workflow built on the reusable ``tsaime`` API."""

from __future__ import annotations

import argparse
import json
import os
import platform
import sys
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Sequence

import numpy as np
import pandas as pd

from tsaime.artifacts import ArtifactWriter
from tsaime.operators import (
    MatrixStandardizer,
    covariance_weighted_inverse,
    fit_inverse_operator,
    marginal_cross_correlation,
    operator_cosine,
    scalar_aime_operator,
)
from tsaime.rolling import RollingVectorTSAIME, RollingVectorTSAIMEConfig
from tsaime.smap import coefficients_in_window_coordinates, run_smap_oos, select_smap_theta
from tsaime.statistics import (
    regression_metrics,
)
from tsaime.temporal import (
    ChronologicalSplit,
    QualityGate,
    audit_hourly_site,
    evaluate_quality_gate,
)



STATE_COLUMNS = (
    "PM25_t",
    "PM25_lag1",
    "PM25_lag24",
    "TEMP",
    "RH",
    "PRESS",
    "RAIN_EVENT",
    "LOG_RAIN",
    "WIND_U",
    "WIND_V",
)


@dataclass(frozen=True)
class APRExperimentConfig:
    run_mode: str = "publication"
    enable_thailand: bool = False
    seed: int = 20260904
    horizons: tuple[int, ...] = (1, 6, 24)
    origin_step_hours: int | None = None
    validation_stride: int | None = None
    theta_grid: tuple[float, ...] | None = None
    inverse_ridge_grid: tuple[float, ...] = (0., .001, .01, .1, 1.)
    forecast_ridge_grid: tuple[float, ...] = (0., .01, .1, 1., 10., 100., 1000., 10000.)
    aime_window_days: int = 28
    aime_window_sensitivity_days: tuple[int, ...] = (14, 28, 56)
    ci_block_days: tuple[int, ...] = (7, 14, 28)
    primary_ci_days: int = 28
    bootstrap_resamples: int | None = None
    null_permutations: int | None = None
    beijing_site_limit: int | None = None
    synthetic_rows_per_regime: int | None = None
    synthetic_regimes: int | None = None
    synthetic_tracking_replicates: int | None = None

    def resolved(self):
        if self.run_mode not in ("publication", "smoke"):
            raise ValueError("run mode must be publication or smoke")
        if self.horizons != (1, 6, 24) or self.enable_thailand:
            raise ValueError("v9.3 freezes the audited 1/6/24h China-Japan design")
        if self.primary_ci_days not in self.ci_block_days:
            raise ValueError("primary CI duration must be included")
        pub=self.run_mode=="publication"
        values=asdict(self)
        defaults=dict(origin_step_hours=6 if pub else 24,validation_stride=4 if pub else 2,
            theta_grid=(0.,.5,1.,2.,4.,8.,16.) if pub else (0.,2.,8.),
            bootstrap_resamples=1999 if pub else 99,beijing_site_limit=12 if pub else 2,
            synthetic_rows_per_regime=600 if pub else 180,synthetic_regimes=20 if pub else 4,
            synthetic_tracking_replicates=20 if pub else 4)
        for key,value in defaults.items():
            if values[key] is None:values[key]=value
        return APRExperimentConfig(**values)


@dataclass(frozen=True)
class APRRunResult:
    """Paths and final status returned to notebooks and command-line callers."""

    run_id: str
    run_root: Path
    output_dir: Path
    zip_path: Path
    readiness: pd.DataFrame
    quality_gate: pd.DataFrame
    forecast_skill: pd.DataFrame


SPLITS = {
    "Beijing": ChronologicalSplit("2013-03-01", "2015-03-01", "2016-03-01", "2017-03-01"),
    "Tsukuba": ChronologicalSplit("2017-01-01", "2020-01-01", "2021-01-01", "2023-01-01"),
    "Thailand": ChronologicalSplit("2021-04-29", "2024-01-01", "2025-01-01", "2026-01-01"),
}


## 5. Shared experiment helpers

In [ ]:
def _regime_season(timestamp: pd.Timestamp) -> str:
    month = timestamp.month
    if month in (12, 1, 2):
        return "DJF"
    if month in (3, 4, 5):
        return "MAM"
    if month in (6, 7, 8):
        return "JJA"
    return "SON"

def _random_correlation(dimension: int, generator: np.random.Generator) -> np.ndarray:
    raw = generator.normal(size=(dimension, dimension))
    covariance = raw @ raw.T
    scale = np.sqrt(np.diag(covariance))
    return covariance / np.outer(scale, scale)

def _baseline_predictions(
    train: pd.DataFrame,
    validation: pd.DataFrame,
    test: pd.DataFrame,
    targets: Sequence[str],
    alpha_grid: Sequence[float],
) -> tuple[dict[str, np.ndarray], pd.DataFrame]:
    """Tune the linear Ridge forecast baseline on validation only."""

    from sklearn.linear_model import Ridge

    train_x_scaler = MatrixStandardizer.fit(train[list(STATE_COLUMNS)].to_numpy(float))
    train_y_scaler = MatrixStandardizer.fit(train[list(targets)].to_numpy(float))
    x_train = train_x_scaler.transform(train[list(STATE_COLUMNS)].to_numpy(float))
    y_train = train_y_scaler.transform(train[list(targets)].to_numpy(float))
    x_validation = train_x_scaler.transform(validation[list(STATE_COLUMNS)].to_numpy(float))
    y_validation = train_y_scaler.transform(validation[list(targets)].to_numpy(float))
    rows = []
    for alpha in alpha_grid:
        model = Ridge(alpha=float(alpha)).fit(x_train, y_train)
        predicted = model.predict(x_validation)
        rows.append(
            {
                "alpha": float(alpha),
                "validation_standardized_RMSE": float(np.sqrt(np.mean((predicted - y_validation) ** 2))),
            }
        )
    selection = pd.DataFrame(rows)
    selected_alpha = float(
        selection.sort_values(["validation_standardized_RMSE", "alpha"]).iloc[0]["alpha"]
    )

    library = pd.concat([train, validation], ignore_index=True)
    x_scaler = MatrixStandardizer.fit(library[list(STATE_COLUMNS)].to_numpy(float))
    y_scaler = MatrixStandardizer.fit(library[list(targets)].to_numpy(float))
    x_library = x_scaler.transform(library[list(STATE_COLUMNS)].to_numpy(float))
    y_library = y_scaler.transform(library[list(targets)].to_numpy(float))
    x_test = x_scaler.transform(test[list(STATE_COLUMNS)].to_numpy(float))
    ridge = Ridge(alpha=selected_alpha).fit(x_library, y_library)
    predictions = {
        "ridge": y_scaler.inverse_transform(ridge.predict(x_test)),
        "persistence": np.repeat(test[["PM25_t"]].to_numpy(float), len(targets), axis=1),
        "training_mean": np.tile(library[list(targets)].mean().to_numpy(float), (len(test), 1)),
    }
    selection["selected"] = selection["alpha"] == selected_alpha
    return predictions, selection

def _notebook_code_sha256(path: Path) -> str:
    payload = json.loads(path.read_text(encoding="utf-8"))
    code = "\n\n".join(
        "".join(cell.get("source", []))
        for cell in payload.get("cells", [])
        if cell.get("cell_type") == "code"
    )
    return hashlib.sha256(code.encode("utf-8")).hexdigest()

def _source_tree_sha256(root: Path) -> str:
    digest = hashlib.sha256()
    for path in sorted(root.rglob("*.py")):
        digest.update(str(path.relative_to(root)).encode("utf-8"))
        digest.update(path.read_bytes())
    return digest.hexdigest()

def _git_worktree_provenance(package_root: Path | None) -> dict[str, object]:
    if package_root is None:
        return {"git_codev9_tracked": "unavailable", "git_codev9_dirty": "unavailable"}
    try:
        tracked = subprocess.run(
            ["git", "ls-files", "--", "src/tsaime", "notebooks/tsAIME_APR_all_experiments_v9_3.ipynb"],
            cwd=package_root, check=True, capture_output=True, text=True,
        ).stdout.strip()
        dirty = subprocess.run(
            ["git", "status", "--porcelain", "--", "."],
            cwd=package_root, check=True, capture_output=True, text=True,
        ).stdout.strip()
        return {
            "git_codev9_tracked": bool(tracked),
            "git_codev9_dirty": bool(dirty),
            "git_codev9_status": dirty,
        }
    except Exception as exc:
        return {
            "git_codev9_tracked": "unavailable",
            "git_codev9_dirty": "unavailable",
            "git_codev9_status": repr(exc),
        }

## 6. Inverse selection, baselines, calendar evaluation and synthetic streams

In [ ]:
from tsaime.calendar_statistics import calendar_skill_intervals, complete_week_shift_diagnostic


def _selection_cut(n: int) -> int:
    if n < 40:
        raise ValueError("at least 40 validation pairs are required")
    return min(max(20, int(n*.65)), n-10)


def _select_inverse_ridge(inputs, outputs, ridge_grid):
    """Unit-invariant tuning on a chronological inner validation holdout."""
    cut = _selection_cut(len(inputs))
    scale = MatrixStandardizer.fit(inputs[:cut]).scale
    rows = []
    for ridge in ridge_grid:
        model = fit_inverse_operator(inputs[:cut], outputs[:cut], ridge=float(ridge))
        loss = np.sqrt(np.mean(((model.reconstruct(outputs[cut:])-inputs[cut:])/scale)**2))
        rows.append(dict(ridge=float(ridge), inverse_RMSE=float(loss),
                         loss_scale="fit_only_standardized_equal_feature_weight", fit_n=cut,
                         selection_n=len(inputs)-cut))
    table = pd.DataFrame(rows)
    best = float(table.sort_values(["inverse_RMSE", "ridge"]).iloc[0].ridge)
    table["selected"] = table.ridge == best
    return best, table


def _quadratic_inverse(inputs, outputs, new_outputs, ridge_grid):
    """Same mean-loss lambda units and X scale as the linear inverse."""
    from sklearn.linear_model import Ridge
    from sklearn.preprocessing import PolynomialFeatures
    cut = _selection_cut(len(inputs))
    xs, ys = MatrixStandardizer.fit(inputs[:cut]), MatrixStandardizer.fit(outputs[:cut])
    poly = PolynomialFeatures(2, include_bias=False)
    design = poly.fit_transform(ys.transform(outputs[:cut]))
    later = poly.transform(ys.transform(outputs[cut:]))
    # Standardize expanded columns, fitted on calibration only.
    ps = MatrixStandardizer.fit(design)
    rows = []
    for lam in ridge_grid:
        model = Ridge(alpha=cut*float(lam)).fit(ps.transform(design), xs.transform(inputs[:cut]))
        loss = np.sqrt(np.mean((model.predict(ps.transform(later))-xs.transform(inputs[cut:]))**2))
        rows.append(dict(ridge=float(lam), validation_inverse_RMSE=float(loss),
                         sklearn_alpha=cut*float(lam), fit_n=cut))
    table = pd.DataFrame(rows)
    lam = float(table.sort_values(["validation_inverse_RMSE", "ridge"]).iloc[0].ridge)
    table["selected"] = table.ridge == lam
    xs, ys = MatrixStandardizer.fit(inputs), MatrixStandardizer.fit(outputs)
    design = poly.fit_transform(ys.transform(outputs)); ps = MatrixStandardizer.fit(design)
    model = Ridge(alpha=len(inputs)*lam).fit(ps.transform(design), xs.transform(inputs))
    prediction = xs.inverse_transform(model.predict(ps.transform(poly.transform(ys.transform(new_outputs)))))
    return prediction, table


def _choose_single_outputs(x, y, ridge_grid):
    """Choose one horizon per input using only the inner validation holdout."""
    cut = _selection_cut(len(x)); xs = MatrixStandardizer.fit(x[:cut])
    losses, lambdas, rows = [], [], []
    for j in range(y.shape[1]):
        lam, _ = _select_inverse_ridge(x, y[:, [j]], ridge_grid)
        model = fit_inverse_operator(x[:cut], y[:cut, [j]], ridge=lam)
        rmse = np.sqrt(np.mean(((model.reconstruct(y[cut:, [j]])-x[cut:])/xs.scale)**2, axis=0))
        losses.append(rmse); lambdas.append(lam)
        for f, loss in enumerate(rmse):
            rows.append(dict(feature=STATE_COLUMNS[f], output_index=j, selected_lambda=lam,
                             validation_RMSE=float(loss)))
    chosen = np.argmin(np.array(losses), axis=0)
    table = pd.DataFrame(rows)
    table["selected_for_feature"] = [r.output_index == chosen[STATE_COLUMNS.index(r.feature)] for r in table.itertuples()]
    return chosen, np.array(lambdas), table


def _inverse_predictions(x, y, new_y, lam, chosen, single_lambdas):
    model = fit_inverse_operator(x, y, ridge=lam)
    marginal = model.y_scaler.transform(new_y) @ model.cross_covariance.T
    pred = {"linear_tsAIME": model.reconstruct(new_y),
            "past_mean": np.tile(model.x_scaler.mean, (len(new_y), 1)),
            "marginal_sum_ablation": model.x_scaler.inverse_transform(marginal),
            "marginal_mean": model.x_scaler.inverse_transform(marginal/y.shape[1])}
    singles = [fit_inverse_operator(x, y[:, [j]], ridge=float(single_lambdas[j])).reconstruct(new_y[:, [j]])
               for j in range(y.shape[1])]
    pred["validation_selected_single"] = np.column_stack([singles[j][:, f] for f, j in enumerate(chosen)])
    for j, value in enumerate(singles):
        pred[f"single_{j}"] = value
    return pred, model


def _comparison_table(dates, obs, pred, base, names, config, *, seed=0, **metadata):
    table = calendar_skill_intervals(dates, obs, pred, base,
        interval_hours=int(config.origin_step_hours), block_days=config.ci_block_days,
        n_resamples=int(config.bootstrap_resamples), seed=config.seed+seed)
    table["variable"] = table.comparison_index.map(dict(enumerate(names)))
    aa = obs[:, None] if obs.ndim == 1 else obs
    pp = pred[:, None] if pred.ndim == 1 else pred
    bb = base[:, None] if base.ndim == 1 else base
    for key, vals in [("RMSE", pp), ("baseline_RMSE", bb)]:
        metric = np.sqrt(np.mean((vals-aa)**2, axis=0))
        table[key] = table.comparison_index.map(dict(enumerate(metric)))
    table["primary_interval"] = table.block_days == config.primary_ci_days
    for key, value in metadata.items():
        table[key] = value
    return table


def _rolling_evaluation(dates, x, y, lam, chosen, single_lambdas, config, *, window_days=None):
    dates = pd.DatetimeIndex(dates)
    width = window_days or config.aime_window_days
    endpoint = dates[0]+pd.Timedelta(days=width)
    minimum = min(30 if config.run_mode == "publication" else 15,
                  max(8, int(.6*width*24/int(config.origin_step_hours))))
    obs, ds, predictions, windows = [], [], {}, []
    while endpoint+pd.Timedelta(days=7) <= dates[-1]:
        fit = np.flatnonzero((dates > endpoint-pd.Timedelta(days=width)) & (dates <= endpoint))
        test = np.flatnonzero((dates > endpoint) & (dates <= endpoint+pd.Timedelta(days=7)))
        if len(fit) >= minimum and len(test) >= 3:
            assert dates[fit].max() < dates[test].min()
            pred, model = _inverse_predictions(x[fit], y[fit], y[test], lam, chosen, single_lambdas)
            obs.append(x[test]); ds.extend(dates[test])
            for name, value in pred.items():
                predictions.setdefault(name, []).append(value)
            windows.append(dict(calibration_start=dates[fit][0], calibration_end=dates[fit][-1],
                                evaluation_start=dates[test][0], evaluation_end=dates[test][-1],
                                calibration_n=len(fit), evaluation_n=len(test), window_days=width,
                                input_mean=model.x_scaler.mean.tolist(), input_scale=model.x_scaler.scale.tolist(),
                                output_mean=model.y_scaler.mean.tolist(), output_scale=model.y_scaler.scale.tolist(),
                                operator=model.operator.tolist()))
        endpoint += pd.Timedelta(days=7)
    if not obs:
        raise RuntimeError("no eligible past-to-next-block inverse windows")
    return pd.DatetimeIndex(ds), np.concatenate(obs), {k: np.concatenate(v) for k,v in predictions.items()}, windows


def _bridge_diagnostic(dates, x, y, coefficients, lam, config):
    dates = pd.DatetimeIndex(dates); rows, operators = [], []
    endpoint = dates[0]+pd.Timedelta(days=config.aime_window_days)
    while endpoint <= dates[-1]:
        ix = np.flatnonzero((dates > endpoint-pd.Timedelta(days=config.aime_window_days)) & (dates <= endpoint))
        if len(ix) >= (30 if config.run_mode == "publication" else 15):
            model = fit_inverse_operator(x[ix], y[ix], ridge=lam)
            xx, yy = model.x_scaler.transform(x[ix]), model.y_scaler.transform(y[ix])
            f = np.median(coefficients_in_window_coordinates(coefficients[ix], x[ix], y[ix]), axis=0)
            residual = yy-xx@f.T
            sigma, cross, noise = xx.T@xx/len(ix), xx.T@residual/len(ix), residual.T@residual/len(ix)
            approximate = covariance_weighted_inverse(f, sigma, ridge=lam, output_noise_covariance=noise)
            full = (sigma@f.T+cross)@np.linalg.pinv(f@sigma@f.T+f@cross+cross.T@f.T+noise+lam*np.eye(y.shape[1]))
            rows.append(dict(endpoint=endpoint, n=len(ix),
                residual_input_cross_covariance_ratio=float(np.linalg.norm(cross)/max(np.linalg.norm(model.cross_covariance), 1e-12)),
                uncorrelated_residual_approximation_cosine=operator_cosine(model.operator, approximate),
                complete_covariance_identity_relative_error=float(np.linalg.norm(full-model.operator)/max(np.linalg.norm(model.operator), 1e-12)),
                interpretation="full expression is an algebraic identity, not independent validation"))
            for i, feature in enumerate(STATE_COLUMNS):
                for j, h in enumerate(config.horizons):
                    operators.append(dict(endpoint=endpoint, season=_regime_season(endpoint), feature=feature,
                        horizon_h=h, AIME=float(model.operator[i,j]), local_input_sd=float(model.x_scaler.scale[i]),
                        local_output_sd=float(model.y_scaler.scale[j])))
        endpoint += pd.Timedelta(days=7)
    return pd.DataFrame(rows), pd.DataFrame(operators)


def _run_site_v93(dataset, site, pairs, targets, config, trace_dir):
    print(f"[{dataset}/{site}] forecast fitting", flush=True)
    train, val, test = [pairs.loc[pairs.split == s].reset_index(drop=True) for s in ("train", "validation", "test")]
    theta, theta_table = select_smap_theta(train, val, STATE_COLUMNS, targets, config.theta_grid,
                                         validation_stride=int(config.validation_stride))
    library = pd.concat([train,val], ignore_index=True)
    sm = run_smap_oos(library, test, STATE_COLUMNS, targets, theta=theta)
    sv = run_smap_oos(train, val, STATE_COLUMNS, targets, theta=theta)
    base, forecast_selection = _baseline_predictions(train, val, test, targets, config.forecast_ridge_grid)
    x, vx, y, vy = test[list(STATE_COLUMNS)].to_numpy(float), val[list(STATE_COLUMNS)].to_numpy(float), sm.predictions, sv.predictions
    obs = test[targets].to_numpy(float); dates = pd.DatetimeIndex(test.Date)
    seed = sum(map(ord, dataset+site))
    tables = {"theta_selection":theta_table, "forecast_ridge_selection":forecast_selection}
    forecasts = {"SMap": y, **base}
    frows=[]
    for name, prediction in forecasts.items():
        frows.append(_comparison_table(dates, obs, prediction, base["persistence"], config.horizons, config,
                                      seed=seed, model=name, baseline="persistence"))
    frows.append(_comparison_table(dates, obs, y, base["ridge"], config.horizons, config,
                                  seed=seed, model="SMap", baseline="tuned_ridge"))
    tables["forecast"] = pd.concat(frows, ignore_index=True).rename(columns={"variable":"horizon_h"})
    lam, tables["inverse_selection"] = _select_inverse_ridge(vx, vy, config.inverse_ridge_grid)
    chosen, single_lams, tables["single_selection"] = _choose_single_outputs(vx, vy, config.inverse_ridge_grid)
    fixed, inverse = _inverse_predictions(vx, vy, y, lam, chosen, single_lams)
    fixed["quadratic_inverse"], tables["quadratic_selection"] = _quadratic_inverse(vx, vy, y, config.inverse_ridge_grid)
    scale = sm.input_scaler
    xz = scale.transform(x)
    inverse_rows=[]
    for name, prediction in fixed.items():
        if name == "linear_tsAIME":
            continue
        inverse_rows.append(_comparison_table(dates, xz, scale.transform(fixed["linear_tsAIME"]),
            scale.transform(prediction), STATE_COLUMNS, config, seed=seed+1000,
            model="linear_tsAIME", baseline=name, evaluation="fixed_validation_to_test"))
    tables["inverse_fixed"] = pd.concat(inverse_rows, ignore_index=True).rename(columns={"variable":"feature"})
    rd, rx, rp, windows = _rolling_evaluation(dates, x, y, lam, chosen, single_lams, config)
    roll_rows=[]
    for name, prediction in rp.items():
        if name == "linear_tsAIME":
            continue
        roll_rows.append(_comparison_table(rd, scale.transform(rx), scale.transform(rp["linear_tsAIME"]),
            scale.transform(prediction), STATE_COLUMNS, config, seed=seed+2000,
            model="linear_tsAIME", baseline=name, evaluation="past_28d_to_next_7d"))
    tables["inverse_rolling"] = pd.concat(roll_rows, ignore_index=True).rename(columns={"variable":"feature"})
    tables["rolling_windows"] = pd.DataFrame(windows).drop(columns=["input_mean","input_scale","output_mean","output_scale","operator"])
    tables["bridge"], tables["operators"] = _bridge_diagnostic(dates, xz, sm.predictions_standardized,
                                                               sm.coefficients_standardized, lam, config)
    tables["null_summary"], tables["null_shifts"] = complete_week_shift_diagnostic(dates, x, y,
        ridge=lam, interval_hours=int(config.origin_step_hours), n_shifts=config.null_permutations, seed=seed)
    # Sensitivity is performance, not merely operator magnitude; all selected lambdas remain frozen.
    sensitivity=[]; window_results={}
    for width in config.aime_window_sensitivity_days:
        dd, xx, pp, _ = _rolling_evaluation(dates,x,y,lam,chosen,single_lams,config,window_days=width)
        window_results[width]=(dd,xx,pp)
    common=None
    for dd,_,_ in window_results.values():
        common=dd if common is None else common.intersection(dd)
    for width,(dd,xx,pp) in window_results.items():
        ix=dd.get_indexer(common)
        for name in ("past_mean","marginal_mean","validation_selected_single"):
            sensitivity.append(_comparison_table(common,scale.transform(xx[ix]),scale.transform(pp["linear_tsAIME"][ix]),
                scale.transform(pp[name][ix]),STATE_COLUMNS,config,seed=seed+3000,window_days=width,baseline=name,
                cohort="common_origins_across_window_lengths"))
    tables["window_sensitivity"] = pd.concat(sensitivity,ignore_index=True).rename(columns={"variable":"feature"})
    print(f"[{dataset}/{site}] meteorology ablations",flush=True)
    ablations=[]; reduced_predictions={}
    feature_sets={"no_pressure":[f for f in STATE_COLUMNS if f!="PRESS"],
                  "no_wind":[f for f in STATE_COLUMNS if f not in ("WIND_U","WIND_V")],
                  "no_pressure_or_wind":[f for f in STATE_COLUMNS if f not in ("PRESS","WIND_U","WIND_V")]}
    selection_rows=[]
    for name, features in feature_sets.items():
        th, st=select_smap_theta(train,val,features,targets,config.theta_grid,validation_stride=int(config.validation_stride))
        st["reduced_model"]=name; st["selected"]=st.theta==th;selection_rows.append(st)
        reduced=run_smap_oos(library,test,features,targets,theta=th).predictions
        reduced_predictions[name]=reduced
        ablations.append(_comparison_table(dates,obs,y,reduced,config.horizons,config,seed=seed+4000,
                                           model="full_SMap",baseline=name))
    tables["meteorology_ablation"]=pd.concat(ablations,ignore_index=True).rename(columns={"variable":"horizon_h"})
    tables["ablation_selection"]=pd.concat(selection_rows,ignore_index=True)
    tables["boundaries"] = pd.DataFrame([dict(split=s, n=len(a), source_start=a.Date.min(),source_end=a.Date.max(),
                                             latest_target=a.Date.max()+pd.Timedelta(hours=max(config.horizons)))
                                           for s,a in [("train",train),("validation",val),("test",test)]])
    trace_dir.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(trace_dir/f"{dataset}_{site}.npz", dates=dates.to_numpy(), inputs=x, observed=obs,
        predictions=y, validation_dates=val.Date.to_numpy(), validation_inputs=vx, validation_predictions=vy,
        library_dates=library.Date.to_numpy(), library_inputs=library[list(STATE_COLUMNS)].to_numpy(float),
        library_targets=library[targets].to_numpy(float), coefficients=sm.coefficients_standardized,
        input_mean=sm.input_scaler.mean,input_scale=sm.input_scaler.scale,
        output_mean=sm.output_scaler.mean,output_scale=sm.output_scaler.scale,
        inverse_operator=inverse.operator,inverse_input_mean=inverse.x_scaler.mean,inverse_input_scale=inverse.x_scaler.scale,
        inverse_output_mean=inverse.y_scaler.mean,inverse_output_scale=inverse.y_scaler.scale,
        inverse_lambda=lam,selected_horizons=chosen,single_lambdas=single_lams,
        **{f"forecast_{k}":v for k,v in base.items()}, **{f"fixed_{k}":v for k,v in fixed.items()},
        rolling_dates=rd.to_numpy(), rolling_inputs=rx, **{f"rolling_{k}":v for k,v in rp.items()},
        **{f"ablation_{k}":v for k,v in reduced_predictions.items()})
    (trace_dir/f"{dataset}_{site}_windows.json").write_text(json.dumps(windows,default=str,indent=2))
    for table in tables.values():
        table.insert(0,"dataset",dataset);table.insert(1,"site",site)
    print(f"[{dataset}/{site}] completed",flush=True)
    return tables


def _synthetic_v93(config):
    """Independent population recovery and a genuinely serial piecewise-linear stream."""
    d,q=len(STATE_COLUMNS),len(config.horizons)
    scalar_rng=np.random.default_rng(config.seed)
    xx=scalar_rng.normal(size=(400,d));yy=scalar_rng.normal(size=400)
    a,_=scalar_aime_operator(xx,yy)
    scalar=pd.DataFrame(dict(feature=STATE_COLUMNS,absolute_difference=np.abs(a-np.array([np.corrcoef(xx[:,j],yy)[0,1] for j in range(d)]))))
    recovery=[]; cross_tasks=[]
    for rep in range(int(config.synthetic_regimes)):
        rng=np.random.default_rng(config.seed+1000+rep)
        sigma=.75*_random_correlation(d,rng)+.25*np.eye(d)
        f=rng.normal(scale=.28,size=(q,d));f[:,:3]+=np.array([[.8,.25,-.1],[.55,.4,.15],[.25,.35,.45]])[:q]
        n=int(config.synthetic_rows_per_regime);x=rng.multivariate_normal(np.zeros(d),sigma,size=n)
        y=x@f.T+rng.normal(scale=.06,size=(n,q));cut=int(.65*n)
        model=fit_inverse_operator(x[:cut],y[:cut],ridge=.01)
        sx,sy=model.x_scaler.scale,model.y_scaler.scale
        fz=f*sx[None,:]/sy[:,None]; noise=np.diag((.06/sy)**2)
        population=covariance_weighted_inverse(fz,sigma/np.outer(sx,sx),ridge=.01,output_noise_covariance=noise)
        xs=model.x_scaler.transform(x);ys=model.y_scaler.transform(y)
        conditional=covariance_weighted_inverse(fz,np.cov(xs[:cut],rowvar=False,ddof=0),ridge=.01,output_noise_covariance=noise)
        dates=pd.date_range("2000-01-01",periods=n,freq="6h")
        frame=pd.DataFrame(x,columns=STATE_COLUMNS);targets=[f"synthetic_{j}" for j in range(q)]
        frame[targets]=y;frame["Date"]=dates
        sf=run_smap_oos(frame.iloc[:cut],frame.iloc[cut:],STATE_COLUMNS,targets,theta=2.)
        fh=np.median(sf.coefficients_standardized,axis=0)
        normal=model.operator@(model.output_covariance+.01*np.eye(q))-model.cross_covariance
        recovery.append(dict(replicate=rep,seed=config.seed+1000+rep,n_fit=cut,
            population_relative_error=float(np.linalg.norm(model.operator-population)/np.linalg.norm(population)),
            sample_conditional_relative_error=float(np.linalg.norm(model.operator-conditional)/np.linalg.norm(conditional)),
            forward_relative_error=float(np.linalg.norm(fh-fz)/np.linalg.norm(fz)),
            normal_equation_error=float(np.linalg.norm(normal)),
            inverse_holdout_RMSE=float(np.sqrt(np.mean((xs[cut:]-ys[cut:]@model.operator.T)**2)))))
        for task,pred,truth,method in [
            ("forward",xs[cut:]@fh.T,ys[cut:],"SMap_forward"),
            ("forward",xs[cut:]@model.operator,ys[cut:],"transposed_inverse_direction_control"),
            ("inverse",ys[cut:]@model.operator.T,xs[cut:],"tsAIME_inverse"),
            ("inverse",ys[cut:]@fh,xs[cut:],"transposed_forward_direction_control")]:
            cross_tasks.append(dict(replicate=rep,task=task,method=method,RMSE=float(np.sqrt(np.mean((pred-truth)**2)))))
    tracking=[];stream_errors=[]
    width=28*4;step=7*4;segment_n=84*4
    for rep in range(int(config.synthetic_tracking_replicates)):
        rng=np.random.default_rng(config.seed+90000+rep)
        sigma=.7*_random_correlation(d,rng)+.3*np.eye(d)
        base=rng.normal(scale=.18,size=(q,d));base[:,:3]+=np.array([[.8,.25,-.1],[.55,.4,.15],[.25,.35,.45]])[:q]
        fs=[]
        for delta in (-.55,-.15,.25,.65):
            ff=base.copy();ff[:,5]+=delta;ff[:,8]-=.7*delta;fs.append(ff)
        n=segment_n*4;regime=np.arange(n)//segment_n
        x=np.zeros((n,d));x[0]=rng.multivariate_normal(np.zeros(d),sigma)
        innovations=rng.multivariate_normal(np.zeros(d),(1-.8**2)*sigma,size=n-1)
        for t in range(1,n):x[t]=.8*x[t-1]+innovations[t-1]
        y=np.einsum("nqd,nd->nq",np.array(fs)[regime],x)+rng.normal(scale=.08,size=(n,q))
        fixed=fit_inverse_operator(x[:width],y[:width],ridge=.01)
        for end in range(width,n-step+1,step):
            aidx=np.arange(end-width,end);bidx=np.arange(end,end+step)
            model=fit_inverse_operator(x[aidx],y[aidx],ridge=.01)
            sx,sy=model.x_scaler.scale,model.y_scaler.scale
            for r in np.unique(regime[bidx]):
                b=bidx[regime[bidx]==r];ff=fs[r]*sx[None,:]/sy[:,None]
                pop=covariance_weighted_inverse(ff,sigma/np.outer(sx,sx),ridge=.01,output_noise_covariance=np.diag((.08/sy)**2))
                reconstruction=model.reconstruct(y[b]);past_mean=np.tile(model.x_scaler.mean,(len(b),1))
                tracking.append(dict(replicate=rep,seed=config.seed+90000+rep,evaluation_start_index=int(b[0]),
                    calibration_end_index=end-1,regime=int(r),days_since_change=float((b[0]-r*segment_n)/4),
                    mixed_calibration_regimes=len(np.unique(regime[aidx]))>1,
                    population_relative_error=float(np.linalg.norm(model.operator-pop)/np.linalg.norm(pop)),
                    estimated_pressure_1h=float(model.operator[5,0]),population_pressure_1h=float(pop[5,0]),
                    forward_pressure_1h=float(ff[0,5]),estimated_wind_1h=float(model.operator[8,0]),
                    population_wind_1h=float(pop[8,0]),
                    next_block_RMSE=float(np.sqrt(np.mean((reconstruction-x[b])**2))),
                    mean_baseline_RMSE=float(np.sqrt(np.mean((past_mean-x[b])**2))),
                    frozen_inverse_RMSE=float(np.sqrt(np.mean((fixed.reconstruct(y[b])-x[b])**2)))))
                for i,t in enumerate(b):
                    stream_errors.append(dict(replicate=rep,index=int(t),regime=int(r),
                        inverse_squared_error=float(np.mean((reconstruction[i]-x[t])**2)),
                        mean_squared_error=float(np.mean((past_mean[i]-x[t])**2))))
    return {"scalar":scalar,"synthetic_recovery":pd.DataFrame(recovery),
            "synthetic_cross_task":pd.DataFrame(cross_tasks),"synthetic_tracking":pd.DataFrame(tracking),
            "synthetic_stream_errors":pd.DataFrame(stream_errors)}


## 7. Policy sensitivity, figures, complete tables and ZIP output

In [ ]:
def _policy_sensitivity_v93(frame, config, trace_dir):
    """Re-fit forecasts and fixed inverses; report policy-specific and common origins.

    Own-policy forecast truth can change; common-origin prediction comparisons
    explicitly use the as-reported truth. Inverse policy results are descriptive
    because both inputs and output representations can change.
    """
    records={};selection=[]
    for policy in PM25_POLICIES:
        _,pairs,_,targets=prepare_pm25_multihorizon_pairs(apply_pm25_policy(frame,policy),config.horizons,
            SPLITS["Tsukuba"],origin_step_hours=int(config.origin_step_hours),required_states=STATE_COLUMNS)
        train,val,test=[pairs[pairs.split==s].reset_index(drop=True) for s in ("train","validation","test")]
        theta,st=select_smap_theta(train,val,STATE_COLUMNS,targets,config.theta_grid,validation_stride=int(config.validation_stride))
        sv=run_smap_oos(train,val,STATE_COLUMNS,targets,theta=theta)
        sm=run_smap_oos(pd.concat([train,val],ignore_index=True),test,STATE_COLUMNS,targets,theta=theta)
        vx=val[list(STATE_COLUMNS)].to_numpy(float);x=test[list(STATE_COLUMNS)].to_numpy(float)
        lam,it=_select_inverse_ridge(vx,sv.predictions,config.inverse_ridge_grid)
        inv=fit_inverse_operator(vx,sv.predictions,ridge=lam)
        records[policy]=dict(dates=pd.DatetimeIndex(test.Date),obs=test[targets].to_numpy(float),pred=sm.predictions,
            persistence=np.repeat(test[["PM25_t"]].to_numpy(float),len(targets),axis=1),
            x=sm.input_scaler.transform(x),recon=sm.input_scaler.transform(inv.reconstruct(sm.predictions)),
            input_scale=sm.input_scaler.scale,baseline=sm.input_scaler.transform(np.tile(vx.mean(axis=0),(len(x),1))),theta=theta,lam=lam)
        st["policy"]=policy;st["selected"]=st.theta==theta;selection.append(st)
        np.savez_compressed(trace_dir/f"Tsukuba_policy_{policy}.npz",**records[policy])
    common=records["as_reported"]["dates"]
    for record in records.values():common=common.intersection(record["dates"])
    rows=[];inverse_rows=[]
    for policy,r in records.items():
        rows.append(_comparison_table(r["dates"],r["obs"],r["pred"],r["persistence"],config.horizons,config,
                    seed=6000,policy=policy,cohort="policy_specific",target="own_policy"))
        idx=r["dates"].get_indexer(common);reference=records["as_reported"]
        ri=reference["dates"].get_indexer(common)
        rows.append(_comparison_table(common,reference["obs"][ri],r["pred"][idx],reference["pred"][ri],
                    config.horizons,config,seed=6000,policy=policy,cohort="common_origins",target="as_reported"))
        inverse_rows.append(_comparison_table(r["dates"],r["x"],r["recon"],r["baseline"],STATE_COLUMNS,config,
                    seed=7000,policy=policy,cohort="policy_specific",target="own_policy_input"))
    return {"pm25_policy_forecast":pd.concat(rows,ignore_index=True).rename(columns={"variable":"horizon_h"}),
            "pm25_policy_inverse":pd.concat(inverse_rows,ignore_index=True).rename(columns={"variable":"feature"}),
            "pm25_policy_theta":pd.concat(selection,ignore_index=True)}


def _figures_v93(tables, writer):
    import matplotlib.pyplot as plt
    plt.switch_backend("Agg")
    plt.rcParams.update({"font.size":10,"pdf.fonttype":42,"ps.fonttype":42})
    recovery=tables["synthetic_recovery"]
    fig,axes=plt.subplots(1,2,figsize=(10,4))
    axes[0].boxplot([recovery.population_relative_error,recovery.sample_conditional_relative_error])
    axes[0].set_xticks([1,2],["Population target","Sample-conditional target"])
    axes[0].set_ylabel("Relative operator error")
    cross=tables["synthetic_cross_task"]
    labels=[];values=[]
    for (task,method),a in cross.groupby(["task","method"]):
        labels.append(task+" / "+method.replace("_"," "));values.append(a.RMSE.median())
    axes[1].barh(labels,values);axes[1].set_xlabel("Median held-out RMSE (standardized)")
    axes[1].tick_params(axis="y",labelsize=7)
    fig.tight_layout();writer.save_figure(fig,"figure01_population_operator_validation")
    tracking=tables["synthetic_tracking"]
    one=tracking[tracking.replicate==0]
    fig,axes=plt.subplots(2,1,figsize=(10,6),sharex=True)
    day=one.evaluation_start_index/4
    axes[0].plot(day,one.population_pressure_1h,label="Current population target")
    axes[0].plot(day,one.estimated_pressure_1h,label="Trailing-window estimate")
    axes[0].set_ylabel("Pressure coefficient\n(local standardized coordinates)");axes[0].legend()
    summary=tracking.groupby("evaluation_start_index").population_relative_error.agg(
        median="median",low=lambda a:a.quantile(.25),high=lambda a:a.quantile(.75))
    axes[1].plot(summary.index/4,summary["median"],label="Replicate median")
    axes[1].fill_between(summary.index/4,summary.low,summary.high,alpha=.2,label="Replicate IQR (not CI)")
    for ax in axes:
        for change in (84,168,252):ax.axvline(change,color="grey",ls=":")
        ax.grid(alpha=.2)
    axes[1].set_ylabel("Population relative error");axes[1].set_xlabel("Stream day");axes[1].legend()
    fig.tight_layout();writer.save_figure(fig,"figure02_serial_regime_tracking")
    forecast=tables["forecast"]
    fig,axes=plt.subplots(1,3,figsize=(12,5),sharey=True)
    for ax,h in zip(axes,(1,6,24)):
        a=forecast[(forecast.horizon_h==h)&(forecast.model=="SMap")&(forecast.baseline=="persistence")&forecast.primary_interval].sort_values(["dataset","site"])
        pos=np.arange(len(a));ax.hlines(pos,a.CI_low*100,a.CI_high*100,color="tab:blue")
        ax.scatter(a.skill*100,pos,s=20);ax.axvline(0,color="grey",ls="--")
        ax.set_yticks(pos,a.site);ax.set_title(f"{h} h ahead");ax.set_xlabel("RMSE improvement (%)")
    fig.suptitle("S-Map vs persistence: pointwise 95% intervals, 28-calendar-day blocks")
    fig.tight_layout();writer.save_figure(fig,"figure03_forecast_skill")
    inverse=tables["inverse_rolling"]
    weather=list(STATE_COLUMNS[3:])
    fig,axes=plt.subplots(1,2,figsize=(12,5),sharey=True)
    for ax,dataset in zip(axes,("Beijing","Tsukuba")):
        for offset,base in enumerate(("past_mean","marginal_mean","validation_selected_single")):
            a=inverse[(inverse.dataset==dataset)&inverse.primary_interval&(inverse.baseline==base)]
            med=a.groupby("feature").skill.median().reindex(weather)
            ax.scatter(med*100,np.arange(len(weather))+(offset-1)*.16,label=base.replace("_"," "),s=28)
        ax.set_yticks(np.arange(len(weather)),weather);ax.axvline(0,color="grey",ls=":")
        ax.set_title(dataset);ax.set_xlabel("Median paired RMSE improvement (%)")
    axes[1].legend(fontsize=8)
    fig.suptitle("Time-local inverse reconstruction; site medians, not confidence intervals")
    fig.tight_layout();writer.save_figure(fig,"figure04_inverse_meteorology_baselines")
    fig,axes=plt.subplots(1,2,figsize=(12,4))
    bridge=tables["bridge"]
    for dataset,a in bridge.groupby("dataset"):
        axes[0].scatter(a.residual_input_cross_covariance_ratio,a.uncorrelated_residual_approximation_cosine,
                        s=8,alpha=.4,label=dataset)
    axes[0].set_xlabel("Omitted cross-covariance / cross-covariance norm")
    axes[0].set_ylabel("Cosine with uncorrelated-residual\napproximation");axes[0].legend()
    ablation=tables["meteorology_ablation"]
    for baseline,a in ablation[ablation.primary_interval].groupby("baseline"):
        med=a.groupby("horizon_h").skill.median();axes[1].plot(med.index,100*med.values,marker="o",label=baseline)
    axes[1].set_xlabel("Forecast horizon (h)");axes[1].set_ylabel("Median full-model RMSE improvement (%)")
    axes[1].set_title("Median across sites; not a pooled estimate")
    axes[1].legend(fontsize=7);axes[1].axhline(0,color="grey",ls=":")
    fig.tight_layout();writer.save_figure(fig,"figure05_bridge_assumption_and_ablation")
    seasonal=tables["operators"].groupby(["dataset","season","feature","horizon_h"],as_index=False).AIME.median()
    fig,axes=plt.subplots(1,2,figsize=(12,5))
    for ax,dataset in zip(axes,("Beijing","Tsukuba")):
        a=seasonal[(seasonal.dataset==dataset)&(seasonal.horizon_h==24)]
        matrix=a.pivot(index="feature",columns="season",values="AIME").reindex(index=weather,columns=["DJF","MAM","JJA","SON"])
        vmax=max(float(np.nanmax(np.abs(matrix.to_numpy()))),.01)
        im=ax.imshow(matrix,aspect="auto",cmap="RdBu_r",vmin=-vmax,vmax=vmax)
        ax.set_xticks(range(4),matrix.columns);ax.set_yticks(range(len(weather)),weather);ax.set_title(dataset)
        fig.colorbar(im,ax=ax,label="Median standardized inverse coefficient")
    fig.suptitle("Descriptive seasonal association with 24 h forecasts; no causal interpretation")
    fig.tight_layout();writer.save_figure(fig,"figure06_seasonal_inverse_profiles")
    return seasonal


def _compact_v93(name, frame):
    """Explicit region summaries; full data always remain in CSV."""
    a=frame.loc[frame.primary_interval] if "primary_interval" in frame else frame
    kind=name.split("_",1)[1] if name.startswith("table") else name
    selections={
        "data_sources":["dataset","doi","version","accessed_on"],
        "license_audit":["dataset","landing_page_license","embedded_file_notice","accessed_on"],
        "download_provenance":["path","bytes","cached","verified_in_run_utc"],
        "quality_gate":["dataset","site","quality_gate","train_coverage","validation_coverage","test_coverage"],
        "boundaries":["dataset","site","split","n","source_start","source_end","latest_target"],
        "computational_checks":["status","run_mode","completed_sites","expected_sites","publication_license_status"],
        "null_summary":["dataset","site","observed_n","full_weeks","shift_tail_rank","phase_mismatch_count"],
        "null_shifts":["dataset","site","shift_weeks","paired_n","operator_norm"],
        "plausibility":["dataset","site","pm25_min","pm25_max","pm25_negative_n"],
        "failures":["dataset","site","error"],
        "scalar":["feature","absolute_difference"],
    }
    if kind in selections:
        return a[[c for c in selections[kind] if c in a]].drop_duplicates().copy()
    if kind=="forecast":
        a=a[a.model=="SMap"]
        return a.groupby(["dataset","baseline","horizon_h"],as_index=False).agg(
            sites=("site","nunique"),median_RMSE=("RMSE","median"),median_skill=("skill","median"),
            positive_pointwise_intervals=("CI_low",lambda v:int((v>0).sum())))
    if kind=="synthetic_cross_task":
        return a.groupby(["task","method"],as_index=False).agg(median_RMSE=("RMSE","median"),
            q25=("RMSE",lambda v:v.quantile(.25)),q75=("RMSE",lambda v:v.quantile(.75)))
    if kind=="synthetic_tracking":
        return a.groupby(["regime","mixed_calibration_regimes"],as_index=False).agg(
            windows=("replicate","size"),median_operator_error=("population_relative_error","median"),
            median_next_RMSE=("next_block_RMSE","median"),median_frozen_RMSE=("frozen_inverse_RMSE","median"))
    if kind in ("operators","seasonal_summary"):
        a=a[(a.horizon_h==24)&a.feature.isin(STATE_COLUMNS[3:])]
        return a.groupby(["dataset","season","feature"],as_index=False).AIME.median()
    if kind=="single_selection":
        return a.loc[a.selected_for_feature,["dataset","site","feature","output_index","selected_lambda","validation_RMSE"]]
    if kind=="bridge":
        return a.groupby("dataset",as_index=False).agg(
            median_cross_covariance_ratio=("residual_input_cross_covariance_ratio","median"),
            median_approximation_cosine=("uncorrelated_residual_approximation_cosine","median"),
            maximum_identity_error=("complete_covariance_identity_relative_error","max"))
    if kind=="theta_selection":
        return a.sort_values(["validation_standardized_RMSE","theta"]).groupby(["dataset","site"],as_index=False).first()[
            ["dataset","site","theta","validation_standardized_RMSE"]]
    if kind=="data_audit":
        return a.groupby(["dataset","year"],as_index=False).agg(sites=("site","nunique"),
            median_complete_coverage=("all_required_coverage","median"),minimum_complete_coverage=("all_required_coverage","min"))
    groups=[key for key in ("dataset","policy","cohort","evaluation","baseline","horizon_h","feature","window_days") if key in a]
    if "skill" in a and groups:
        result=a.groupby(groups,dropna=False,as_index=False).agg(median_skill=("skill","median"),
            positive_pointwise_intervals=("CI_low",lambda x:int((x>0).sum())),comparisons=("skill","size"))
        # A complete longtable is used by the writer below; never silently truncate rows.
        return result
    if "selected" in a:
        chosen=a.loc[a.selected]
        return chosen.drop(columns=[c for c in ("loss_scale","selected") if c in chosen])
    numeric=[c for c in a.select_dtypes(include=np.number).columns if c not in ("replicate","seed","index","comparison_index")]
    if not numeric:
        return a.melt(var_name="field",value_name="value")
    return pd.DataFrame({"metric":numeric,"median":[a[c].median() for c in numeric],
                         "q25":[a[c].quantile(.25) for c in numeric],"q75":[a[c].quantile(.75) for c in numeric]})


def _save_table_v93(writer, frame, stem, caption):
    csv_path=writer.save_csv(frame,stem+".csv")
    compact=_compact_v93(stem,frame).replace([np.inf,-np.inf],np.nan)
    # Compact presentation does not alter the machine-readable CSV schema.
    compact=compact.replace({"validation_selected_single":"Selected single",
        "marginal_sum_ablation":"Correlation sum", "marginal_mean":"Correlation mean",
        "past_mean":"Past mean", "quadratic_inverse":"Quadratic",
        "fixed_validation_to_test":"Fixed", "past_28d_to_next_7d":"Rolling",
        "no_pressure_or_wind":"No pressure/wind", "no_pressure":"No pressure", "no_wind":"No wind",
        "policy_specific":"Policy-specific", "common_origins":"Common origins"})
    for col in ("median_skill",):
        if col in compact: compact[col]=100*compact[col]
    compact=compact.rename(columns={"dataset":"Region","feature":"Input","horizon_h":"h",
        "evaluation":"Evaluation","baseline":"Reference","window_days":"Window (d)",
        "median_skill":"Skill (percent)","positive_pointwise_intervals":"CI above 0",
        "comparisons":"N","policy":"Policy","cohort":"Cohort",
        "validation_standardized_RMSE":"Validation RMSE","validation_inverse_RMSE":"Validation RMSE",
        "inverse_RMSE":"Validation RMSE","selected_lambda":"Lambda","selected_for_feature":"Chosen",
        "forecast_ridge_upper_boundary_sites":"Upper bound fits"})
    for col in compact.select_dtypes(include=["object","string"]).columns:
        compact[col]=compact[col].map(lambda v:v.replace("AMEL.hourly.","AMEL ").replace(".Ver1.0.txt"," (v1.0)").replace("_"," ") if isinstance(v,str) else v)
    compact.columns=[str(c).replace("_"," ") for c in compact.columns]
    # Long metric names/URLs wrap inside fixed-width columns rather than run off the page.
    column_format="".join("p{"+str(round(.92/max(len(compact.columns),1),4))+"\\linewidth}" for _ in compact.columns)
    # longtable is page-breakable and includes ALL rows of the defined summary.
    tex=compact.to_latex(index=False,escape=True,longtable=True,
                          float_format=lambda v:f"{v:.3e}" if 0<abs(v)<1e-4 else f"{v:.4f}",
                          na_rep="--",column_format=column_format,
                          caption=caption,label="tab:"+stem)
    (writer.tex_dir/(stem+".tex")).write_text(
        "% Requires \\usepackage{booktabs,longtable}\n"
        +f"% Complete CSV: {len(frame)} rows. Explicit summary: {len(compact)} rows.\n"
        +"\\begingroup\\small\\setlength{\\tabcolsep}{2pt}\n"+tex+"\\endgroup\n")
    return csv_path


def run_apr_pm25(base_dir, config=None):
    config=(config or APRExperimentConfig()).resolved()
    base=Path(base_dir).expanduser().resolve()
    run_id=datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    run_root=base/"results"/"v9_3_apr"/run_id
    run_root.mkdir(parents=True,exist_ok=False)
    os.environ.setdefault("MPLBACKEND","Agg")
    os.environ.setdefault("MPLCONFIGDIR",str(run_root/".matplotlib"))
    writer=ArtifactWriter(run_root/"outputs",clear_existing=False)
    trace=run_root/"private_traces"
    trace.mkdir()
    cache=Path(os.environ.get("TSAIME_DATA_CACHE",str(base/"results"/"v9_apr"/"data"))).expanduser()
    repository=PublicDataRepository(cache)
    notebook_path=PACKAGE_ROOT/"notebooks"/"tsAIME_APR_all_experiments_v9_3.ipynb" if PACKAGE_ROOT else None
    info=dict(workflow="APR v9.3",run_id=run_id,package_version=__import__("tsaime").__version__,
        config=asdict(config),python=sys.version,platform=platform.platform(),
        dependencies={p:importlib.metadata.version(p) for p in ("numpy","pandas","scipy","scikit-learn","matplotlib","pyEDM")},
        notebook_code_sha256=_notebook_code_sha256(notebook_path) if notebook_path else "unavailable",
        package_source_sha256=_source_tree_sha256(PACKAGE_ROOT/"src"/"tsaime") if PACKAGE_ROOT else "unavailable",
        protocol="post-v9.2 audit correction; not preregistered unseen confirmation",
        interval="pointwise; fitted-model conditional; noncircular calendar MBB",
        cache_directory=str(cache.resolve()),**_git_worktree_provenance(PACKAGE_ROOT))
    (writer.log_dir/"run_info.json").write_text(json.dumps(info,default=str,indent=2))
    print("Run:",run_root,flush=True)
    print("Synthetic population and serial validation",flush=True)
    tables=_synthetic_v93(config)
    frames={("Beijing",site):frame for site,frame in repository.load_beijing_sites(int(config.beijing_site_limit)).items()}
    frames[("Tsukuba","NIES_Tsukuba")]=repository.load_tsukuba()
    if config.enable_thailand:
        raise ValueError("v9.3 freezes the audited China-Japan design; optional extensions require a separate protocol")
    audits=[];gates=[];plausibility=[];results={};errors=[]
    gate=QualityGate(minimum_rows={"train":1000,"validation":500,"test":500} if config.run_mode=="publication"
                     else {"train":100,"validation":50,"test":50},minimum_coverage=.60)
    for (dataset,site),frame in frames.items():
        start,_,_,end=SPLITS[dataset].timestamps()
        audits.append(audit_hourly_site(frame,["PM25","TEMP","RH","PRESS","RAIN","WIND_U","WIND_V"],
                                        dataset=dataset,site=site,start=start,end=end))
        plausibility.append(pm25_plausibility_audit(frame,dataset=dataset,site=site))
        candidates,pairs,states,targets=prepare_pm25_multihorizon_pairs(frame,config.horizons,SPLITS[dataset],
            origin_step_hours=int(config.origin_step_hours),required_states=STATE_COLUMNS)
        q=evaluate_quality_gate(candidates,pairs,gate,dataset=dataset,site=site);gates.append(q)
        if q["quality_gate"]!="PASS":
            errors.append(dict(dataset=dataset,site=site,error="data quality gate failed"));continue
        try:
            result=_run_site_v93(dataset,site,pairs,targets,config,trace)
            results[(dataset,site)]=result
            # Per-site checkpoints survive a later site's error or kernel interruption.
            for name,part in result.items():writer.save_csv(part,f"site_{site}_{name}.csv")
        except Exception as exc:
            import traceback
            errors.append(dict(dataset=dataset,site=site,error=repr(exc)))
            (writer.log_dir/f"error_{site}.txt").write_text(traceback.format_exc())
            print("FAILED:",site,repr(exc),flush=True)
    if not results:
        pd.DataFrame(errors).to_csv(writer.csv_dir/"failures.csv",index=False)
        raise RuntimeError("No site completed; see logs and failures.csv in "+str(writer.root))
    for name in next(iter(results.values())):
        tables[name]=pd.concat([r[name] for r in results.values()],ignore_index=True)
    print("Tsukuba negative-PM2.5 sensitivity",flush=True)
    try:tables.update(_policy_sensitivity_v93(frames[("Tsukuba","NIES_Tsukuba")],config,trace))
    except Exception as exc:
        errors.append(dict(dataset="Tsukuba",site="policy_sensitivity",error=repr(exc)))
    executed_regions={dataset for dataset,site in frames}
    used_sources=source_table().loc[lambda a:a.dataset.isin(executed_regions)].copy()
    tables.update(data_sources=used_sources,license_audit=pd.DataFrame(repository.license_records),
        download_provenance=pd.DataFrame(repository.records),data_audit=pd.concat(audits,ignore_index=True),
        quality_gate=pd.DataFrame(gates),plausibility=pd.DataFrame(plausibility),
        failures=pd.DataFrame(errors,columns=["dataset","site","error"]))
    tables["seasonal_summary"]=_figures_v93(tables,writer)
    expected=int(config.beijing_site_limit)+1
    computational_ok=len(results)==expected and not errors and tables["scalar"].absolute_difference.max()<1e-10
    computational_ok &= tables["synthetic_recovery"].normal_equation_error.max()<1e-8
    computational_ok &= tables["null_summary"].phase_mismatch_count.max()==0
    computational_ok &= (tables["synthetic_tracking"].calibration_end_index<tables["synthetic_tracking"].evaluation_start_index).all()
    readiness=pd.DataFrame([dict(status="PASS" if computational_ok else "FAIL",run_id=run_id,run_mode=config.run_mode,
        completed_sites=len(results),expected_sites=expected,failed_tasks=len(errors),
        publication_verification="full_scale_run" if config.run_mode=="publication" else "smoke_only",
        publication_license_status="manual_NIES_notice_resolution_required",
        scientific_conclusion="not gated on significance, model wins, or small recovery error")])
    tables["computational_checks"]=readiness
    captions={"forecast":"S-Map forecast skill summary; full CSV also includes other models. Pointwise fitted-model conditional 95 percent intervals; primary block length 28 calendar days.",
        "operators":"Seasonal median inverse coefficients for meteorological inputs and the 24-hour forecast; full CSV contains all variables, horizons, and windows.",
        "seasonal_summary":"Seasonal median meteorological inverse coefficients for the 24-hour forecast; full CSV contains all variables and horizons.",
        "inverse_fixed":"Fixed inverse reconstruction; positive skill favors linear ts-AIME over the named baseline.",
        "inverse_rolling":"Past 28-day to next 7-day inverse reconstruction; positive skill favors linear ts-AIME.",
        "synthetic_recovery":"Population operator recovery and separately labeled sample-conditional consistency.",
        "synthetic_tracking":"Serial AR(1) regime-change experiment; calibration strictly precedes evaluation.",
        "null_summary":"Complete-week shift diagnostic; tail ranks are not exact p-values under seasonal nonstationarity.",
        "bridge":"Uncorrelated-residual approximation and full covariance algebraic identity. Not independent validation."}
    mapping=[]
    for number,(name,frame) in enumerate(tables.items(),start=1):
        stem=f"table{number:02d}_{name}";caption=captions.get(name,name.replace("_"," ").capitalize()+"; see the protocol for interpretation.")
        _save_table_v93(writer,frame,stem,caption);mapping.append(dict(table=stem,analysis=name,rows=len(frame)))
    writer.save_csv(pd.DataFrame(mapping),"TABLE_INDEX.csv")
    trace_index=[]
    for path in sorted(trace.glob("*")):
        trace_index.append(dict(file=path.name,bytes=path.stat().st_size,sha256=_sha256(path)))
    writer.save_csv(pd.DataFrame(trace_index),"PRIVATE_TRACE_MANIFEST.csv")
    writer.write_text("README_RESULTS.md",f"""# Time Series AIME v9.3 run {run_id}

Mode: {config.run_mode}. Computational checks: {readiness.iloc[0]['status']}.
PASS means completion and mathematical/temporal checks, not acceptance or positive findings.
Publication-mode and smoke-mode outputs must never be pooled.
Primary confidence intervals use 28-calendar-day noncircular moving blocks; 7 and 14 days are sensitivity analyses.
Intervals are pointwise conditional on fitted predictions, not simultaneous or refit uncertainty.
Inverse lambda and single horizons are selected on validation only with fit-only standardized loss.
Linear and quadratic lambda both penalize the mean squared residual objective.
Rolling inverse calibration uses past test-period inputs; S-Map library remains fixed before test.
Population and sample-conditional synthetic targets are separate. Serial tracking uses a genuine AR(1) stream.
Full covariance bridge closure is an algebraic identity, not independent scientific evidence.
Shift tail ranks require shift-invariance for a test interpretation and do not control seasonal confounding.
Sites are not independent geographic regions. This is not China-to-Japan model transfer.
Negative-PM2.5 sensitivity includes policy-specific and common-origin forecast results.
NIES license notice conflict remains a manual release blocker. This software license does not relicense data.
Private traces containing observations are stored outside this ZIP in ../private_traces; do not redistribute without checking dataset terms.
TABLE_INDEX.csv maps every complete CSV to its analysis. TeX files use booktabs and longtable.
""")
    writer.write_text("DATA_AVAILABILITY.txt","Sources: UCI DOI 10.24432/C5RK5G; NIES DOI 10.17595/20250418.001, Version 1.0. "
        "Original source snapshot metadata: 2026-09-04. File checksums and verification times are in download_provenance. "
        "No raw observations are redistributed in this archive. NIES embedded and landing-page notices conflict; resolve before release.")
    writer.manifest();zip_path=writer.create_zip(f"tsAIME_APR_v9_3_outputs_{run_id}.zip")
    print("Completed:",zip_path,flush=True)
    return APRRunResult(run_id,run_root,writer.root,zip_path,readiness,tables["quality_gate"],tables["forecast"])


## 8. Confirm the run mode

Publication mode is the default. Smoke results are not manuscript results.

In [ ]:
RUN_MODE = os.environ.get("TSAIME_V9_RUN_MODE", "publication")
CONFIG = APRExperimentConfig(run_mode=RUN_MODE).resolved()
print(CONFIG)
display(source_table())
display(pd.DataFrame({"input":STATE_COLUMNS}))

## 9. Run all experiments

Each run writes a separate timestamped directory. No old result is erased. Open the result README and computational checks before interpreting findings.

In [ ]:
RESULT = run_apr_pm25(RESULT_BASE_DIR, CONFIG)
display(RESULT.readiness)
print("Output directory:", RESULT.output_dir)
print("Result ZIP:", RESULT.zip_path)

## 10. Inspect forecast performance

Positive skill is RMSE reduction, not percentage accuracy. Beijing and Tsukuba were fitted independently.

In [ ]:
FORECAST = RESULT.forecast_skill
display(FORECAST.loc[FORECAST.primary_interval & (FORECAST.model == "SMap") & (FORECAST.baseline == "persistence")].groupby(["dataset","horizon_h"]).agg(median_skill=("skill","median"),positive_pointwise_intervals=("CI_low",lambda a:int((a>0).sum()))))